# NYC full pipeline (extended audio)

**Flow:** (1) Resample the full mix to the UNet sample rate and run **the same STFT + UNet** as `separate_voice`, but keep only a per-frame **mask×magnitude** activity curve (no full-length `voice_est` / ISTFT). (2) **Flag** high-activity regions, export **10 s** windows. (3) Run **`anonymize_audio` on each crop only** (local STFT boundaries—see caveat). (4) **Splice** each crop’s `anonymized_mix` back into a full-length output per blur mode.

**Caveat:** Detection uses the full timeline; separation inside `anonymize_audio` on each 10 s clip is **local** and not bitwise-identical to a hypothetical single full-file ISTFT.


In [5]:
from __future__ import annotations

import shutil
import sys
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "anonymization_pipeline").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "anonymization_pipeline").is_dir():
    raise RuntimeError("Run this notebook from the repo root or from repo_root/notebooks")

sys.path.insert(0, str(REPO_ROOT))

from anonymization_pipeline import anonymize_audio
from source_separation import (
    compute_voice_activity,
    crop_windows_from_activity_with_config,
    load_unet_checkpoint,
    splice_anonymized_segments,
)


In [6]:
AUDIO_PATH = REPO_ROOT / (
    "notebooks/New York City Streets - Real Sounds & Urban Atmosphere "
    "\uff5c Walking Tour \uff5c 4K.wav"
)
CKPT_PATH = REPO_ROOT / "checkpoints/unet_run3/unet_voice_sep.pt"
OUTPUT_DIR = REPO_ROOT / "notebooks" / "nyc_pipeline_outputs"
BLUR_MODES = ["low_pass", "mfcc"]

# Activity → windows (tune if you get too many / too few crops)
ACTIVITY_QUANTILE = 0.98
MIN_RUN_FRAMES = 3
MERGE_GAP_FRAMES = 8
WINDOW_SEC = 10.0
MAX_WINDOWS = 48

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"Missing input audio: {AUDIO_PATH}")
if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUDIO_PATH, OUTPUT_DIR / "full_input.wav")

y_full, sr_loaded = librosa.load(str(AUDIO_PATH), sr=None, mono=True)
y_full = np.asarray(y_full, dtype=np.float32)
model, config, device = load_unet_checkpoint(CKPT_PATH, device="auto")
sr_out = int(config.sample_rate)
if sr_loaded != sr_out:
    y_full = librosa.resample(y_full, orig_sr=sr_loaded, target_sr=sr_out).astype(np.float32)

activity, sr_act = compute_voice_activity(y_full, sr_out, model, config, device)
assert sr_act == sr_out

sf.write(OUTPUT_DIR / "full_mix_16k.wav", y_full, sr_out)

windows = crop_windows_from_activity_with_config(
    activity,
    config,
    total_samples=len(y_full),
    window_sec=WINDOW_SEC,
    quantile=ACTIVITY_QUANTILE,
    min_run_frames=MIN_RUN_FRAMES,
    merge_gap_frames=MERGE_GAP_FRAMES,
    max_windows=MAX_WINDOWS,
)
print(f"sr_out={sr_out} | samples={len(y_full)} | activity len={len(activity)} | n_windows={len(windows)}")
for i, (a, b) in enumerate(windows[:12]):
    print(f"  crop {i}: samples [{a}:{b}] ({(b-a)/sr_out:.2f}s)")
if len(windows) > 12:
    print("  ...")


sr_out=16000 | samples=35483806 | activity len=138609 | n_windows=15
  crop 0: samples [638336:798336] (10.00s)
  crop 1: samples [3884672:4044672] (10.00s)
  crop 2: samples [5499520:5664896] (10.34s)
  crop 3: samples [13128064:13338496] (13.15s)
  crop 4: samples [14372480:14532480] (10.00s)
  crop 5: samples [15174528:15334528] (10.00s)
  crop 6: samples [18040960:18200960] (10.00s)
  crop 7: samples [27345536:27540608] (12.19s)
  crop 8: samples [28289152:28569984] (17.55s)
  crop 9: samples [29806464:30100352] (18.37s)
  crop 10: samples [30207360:31049088] (52.61s)
  crop 11: samples [31148672:31308672] (10.00s)
  ...


In [7]:
segments_by_mode = {m: [] for m in BLUR_MODES}
nw = len(windows)

for i, (s0, s1) in enumerate(windows):
    y_crop = y_full[s0:s1].astype(np.float32)
    ttag = f"{s0 / sr_out:.1f}s"
    sf.write(OUTPUT_DIR / f"crop_{i:03d}_t{ttag}_mix.wav", y_crop, sr_out)
    for mode in BLUR_MODES:
        res = anonymize_audio(
            y_crop,
            sr_out,
            model=model,
            config=config,
            device=device,
            blur_mode=mode,
        )
        sf.write(
            OUTPUT_DIR / f"crop_{i:03d}_t{ttag}_anonymized_{mode}.wav",
            res.anonymized_mix,
            res.sr,
        )
        seg = np.asarray(res.anonymized_mix, dtype=np.float32).reshape(-1)
        segments_by_mode[mode].append((s0, seg))
    print(f"crop {i + 1}/{nw} (t={ttag})")

for mode in BLUR_MODES:
    segments_by_mode[mode].sort(key=lambda x: x[0])

for mode in BLUR_MODES:
    assembled = splice_anonymized_segments(
        y_full, segments_by_mode[mode], crossfade_samples=256
    )
    out_wav = OUTPUT_DIR / f"full_anonymized_{mode}.wav"
    sf.write(out_wav, assembled, sr_out)
    print(f"Wrote {out_wav.name}")


crop 1/15 (t=39.9s)
crop 2/15 (t=242.8s)
crop 3/15 (t=343.7s)
crop 4/15 (t=820.5s)
crop 5/15 (t=898.3s)
crop 6/15 (t=948.4s)
crop 7/15 (t=1127.6s)
crop 8/15 (t=1709.1s)
crop 9/15 (t=1768.1s)
crop 10/15 (t=1862.9s)
crop 11/15 (t=1888.0s)
crop 12/15 (t=1946.8s)
crop 13/15 (t=1975.4s)
crop 14/15 (t=2059.5s)
crop 15/15 (t=2202.0s)
Wrote full_anonymized_low_pass.wav
Wrote full_anonymized_mfcc.wav


In [8]:
# Example: first flagged crop — mix from disk + anonymized written in previous cell
if windows:
    s0, s1 = windows[0]
    ttag = f"{s0 / sr_out:.1f}s"
    mix_p = OUTPUT_DIR / f"crop_000_t{ttag}_mix.wav"
    ano_p = OUTPUT_DIR / f"crop_000_t{ttag}_anonymized_{BLUR_MODES[0]}.wav"
    xm, _ = sf.read(mix_p, dtype="float32", always_2d=False)
    xa, sra = sf.read(ano_p, dtype="float32", always_2d=False)
    print("First crop — input mix @ model SR")
    display(Audio(xm, rate=sr_out))
    print(f"Same crop — saved anonymized ({BLUR_MODES[0]})")
    display(Audio(xa, rate=sra))
else:
    print("No windows above threshold; lower ACTIVITY_QUANTILE or MIN_RUN_FRAMES.")


First crop — input mix @ model SR


Same crop — saved anonymized (low_pass)
